In [71]:
import pandas as pd

# helps with fuzzy mergine - AI help

import subprocess
subprocess.run(["pip", "install", "thefuzz", "python-Levenshtein"])

CompletedProcess(args=['pip', 'install', 'thefuzz', 'python-Levenshtein'], returncode=0)

In [72]:
# define fuzzy merge - AI help
# Run this once: pip install thefuzz python-Levenshtein
from thefuzz import process
import pandas as pd  # Added missing pandas import

def fuzzy_merge(df_left, df_right, key, threshold=80):
    """
    Merges two dataframes using fuzzy string matching on 'key' column.
    threshold: match score 0-100, higher = stricter. 80 is a good default.
    """
    # Get unique country names from both sides
    right_countries = df_right[key].unique().tolist()
    
    # For each country in left, find best match in right
    matches = []
    for country in df_left[key].unique():
        # Fixed: unpack only 2 values instead of 3
        match, score = process.extractOne(country, right_countries)
        if score >= threshold:
            matches.append({key: country, key + "_matched": match, "score": score})
        else:
            matches.append({key: country, key + "_matched": None, "score": score})
    
    match_df = pd.DataFrame(matches)
    
    # Merge match table onto left df
    df_left = df_left.merge(match_df, on=key, how="left")
    
    # Rename matched key in right df for joining
    df_right = df_right.rename(columns={key: key + "_matched"})
    
    # Final merge on matched key
    result = df_left.merge(df_right, on=key + "_matched", how="left")
    result = result.drop(columns=[key + "_matched", "score"])
    
    return result

In [49]:
#load data
violence_data = pd.read_excel("../data/Violence Data.xlsx")

In [50]:
state_religion_data = pd.read_excel("../data/state_religions.xlsx")

In [51]:
print(violence_data.head())
violence_data.shape

   country_id_cy   country_cy  year_cy region_cy          main_govt_name_cy  \
0            700  Afghanistan     1989      Asia  Government of Afghanistan   
1            700  Afghanistan     1990      Asia  Government of Afghanistan   
2            700  Afghanistan     1991      Asia  Government of Afghanistan   
3            700  Afghanistan     1992      Asia  Government of Afghanistan   
4            700  Afghanistan     1993      Asia  Government of Afghanistan   

   sb_exist_cy  sb_dyad_count_cy           sb_dyad_ids_cy  \
0            1                 5  724; 726; 727; 729; 732   
1            1                 5  724; 726; 727; 732; 733   
2            1                 4       724; 726; 727; 732   
3            1                 4       724; 726; 727; 732   
4            1                 4       726; 732; 734; 842   

                                    sb_dyad_names_cy  sb_deaths_parties_cy  \
0  Government of Afghanistan - Hizb-i Islami-yi A...                  1019   
1 

(6936, 74)

In [52]:
# filter for intrastate violence

violence_data2 = violence_data[violence_data["sb_intrastate_exist_cy"] == 1]

violence_filtered = violence_data2[[
    "country_cy",
    "year_cy",
    "sb_intrastate_deaths_best_cy",
    "sb_intrastate_deaths_civilians_cy"
]]
violence_filtered.head()

,country_cy,year_cy,sb_intrastate_deaths_best_cy,sb_intrastate_deaths_civilians_cy
0,Afghanistan,1989,5174,303
1,Afghanistan,1990,1478,101
2,Afghanistan,1991,3302,38
3,Afghanistan,1992,4287,1687
4,Afghanistan,1993,4071,611


In [53]:
# remane columns for clarity
violence_filtered = violence_filtered.rename(columns={
    "country_cy": "country",
    "year_cy": "year",
    "sb_intrastate_deaths_best_cy": "intrastate_deaths",
    "sb_intrastate_deaths_civilians_cy": "civilian_deaths"
})
violence_filtered.head()

,country,year,intrastate_deaths,civilian_deaths
0,Afghanistan,1989,5174,303
1,Afghanistan,1990,1478,101
2,Afghanistan,1991,3302,38
3,Afghanistan,1992,4287,1687
4,Afghanistan,1993,4071,611


In [54]:
print(state_religion_data.head())
state_religion_data.shape
print(state_religion_data.columns)

   Unnamed: 0  ccode        cname  year  ccode_qog    cname_qog ccodealp  \
0          46    4.0  Afghanistan  1992          4  Afghanistan      AFG   
1          47    4.0  Afghanistan  1993          4  Afghanistan      AFG   
2          48    4.0  Afghanistan  1994          4  Afghanistan      AFG   
3          49    4.0  Afghanistan  1995          4  Afghanistan      AFG   
4          50    4.0  Afghanistan  1996          4  Afghanistan      AFG   

   ccodecow        cname_year ccodealp_year  biu_offrel  
0     700.0  Afghanistan 1992         AFG92           2  
1     700.0  Afghanistan 1993         AFG93           2  
2     700.0  Afghanistan 1994         AFG94           2  
3     700.0  Afghanistan 1995         AFG95           2  
4     700.0  Afghanistan 1996         AFG96           2  
Index(['Unnamed: 0', 'ccode', 'cname', 'year', 'ccode_qog', 'cname_qog',
       'ccodealp', 'ccodecow', 'cname_year', 'ccodealp_year', 'biu_offrel'],
      dtype='object')


In [55]:
# filter state religion data

religion_filtered = state_religion_data[[
    "cname",
    "biu_offrel"
]]
religion_filtered.head()

,cname,biu_offrel
0,Afghanistan,2
1,Afghanistan,2
2,Afghanistan,2
3,Afghanistan,2
4,Afghanistan,2


In [56]:
# and rename

religion_filtered = religion_filtered.rename(columns={
    "cname": "country",
    "biu_offrel": "state-sponsored religion index"
})


In [57]:
# drop duplicate countries to get rid of time variable

religion_filtered2 = religion_filtered.drop_duplicates(
    subset=["country"]
)

religion_filtered2.head()

,country,state-sponsored religion index
0,Afghanistan,2
23,Albania,0
48,Algeria,2
73,Andorra,0
98,Angola,0


In [75]:
# load third dataset

majority_data = pd.read_csv("../data/religious_data.csv")
majority_data.head()

,Region,Country,Year,Diversity_rank,RDI_score,Diveristy_level,Buddhists,Christians,Hindus,Jews,Muslims,Other_Religions,Religiously_Unaffiliated,Level,Countrycode
0,World,All World,2010,NaN,8.982457,Very high,4.882030,30.584753,14.977995,0.198187,23.869791,2.195080,23.292166,3,900
1,World,All World,2020,NaN,8.966722,Very high,4.111001,28.771444,14.936468,0.187421,25.648489,2.183286,24.161890,3,900
2,Asia-Pacific,All Asia-Pacific,2010,NaN,8.804169,Very high,8.146677,6.138703,25.278584,0.004395,24.763258,2.649087,33.019299,2,90001
3,Asia-Pacific,All Asia-Pacific,2020,NaN,8.738061,Very high,6.955449,5.915316,25.671322,0.004072,26.132376,2.476142,32.845325,2,90001
4,Europe,All Europe,2010,NaN,4.724689,Moderate,0.271246,74.643921,0.222803,0.187239,5.295826,0.673520,18.705446,2,90002



The `process.extractOne()` function typically returns a tuple of `(match, score)` when a match is found, but the code expects 3 values. This can happen depending on the version of the library or when no match is found.

Would you like me to provide the corrected code?

In [78]:
print("violence_filtered rows before merge:", violence_filtered.shape[0])
print("religion_filtered2 rows:", religion_filtered2.shape[0])

violence_filtered rows before merge: 1453
religion_filtered2 rows: 177


In [77]:
# fuzzy merge - AI help
final_data = fuzzy_merge(
    violence_filtered,
    religion_filtered2,
    key="country",
    threshold=80
)

print("Missing state religion after fuzzy merge:", 
      final_data["state-sponsored religion index"].isna().sum())

print("final_data rows after merge:", final_data.shape[0])
print("Missing state religion:", final_data["state-sponsored religion index"].isna().sum())

# ── STEP 2: fuzzy merge + majority data ─────────────────────────
final_data2 = fuzzy_merge(
    final_data,
    majority_filtered,
    key="country",
    threshold=80
)

print("Missing majority religion after fuzzy merge:", 
      final_data2["majority_religion"].isna().sum())
print("Total observations:", final_data2.shape[0])

Missing state religion after fuzzy merge: 43
final_data rows after merge: 1453
Missing state religion: 43
Missing majority religion after fuzzy merge: 0
Total observations: 1453



The key changes made:
1. **Fixed the unpacking issue**: Changed `match, score, _ = process.extractOne(country, right_countries)` to `match, score = process.extractOne(country, right_countries)` since the function only returns 2 values
2. **Added missing import**: Added `import pandas as pd` which was missing from the original code

In [60]:
# filter only for 2020

majority_data2 = majority_data[majority_data["Year"] == 2020]
majority_data2 = majority_data2.rename(columns={
    "Country": "country"
})
majority_data3 = majority_data2.iloc[7:].copy()

majority_data3.head()

,Region,country,Year,Diversity_rank,RDI_score,Diveristy_level,Buddhists,Christians,Hindus,Jews,Muslims,Other_Religions,Religiously_Unaffiliated,Level,Countrycode
15,Asia-Pacific,Afghanistan,2020,200.0,0.032175,Very low,0.020001,0.019379,0.000128,0.000026,99.861969,0.090043,0.008457,1,4
17,Europe,Albania,2020,61.0,4.751494,Moderate,0.000158,17.815659,0.000708,0.010050,74.507225,0.013316,7.652888,1,8
19,Middle East-North Africa,Algeria,2020,182.0,0.372425,Very low,0.015001,0.294990,0.000000,0.000130,98.382347,0.041329,1.266207,1,12
21,Sub-Saharan Africa,Angola,2020,143.0,1.522381,Low,0.002393,93.044708,0.009274,0.001197,0.255656,0.571968,6.114808,1,24
23,Latin America-Caribbean,Argentina,2020,118.0,2.436827,Moderate,0.031062,88.453941,0.002574,0.384978,0.929196,0.965338,9.232909,1,32


In [61]:
religion_cols = [
    "Buddhists",
    "Christians",
    "Hindus",
    "Jews",
    "Muslims",
    "Other_Religions",
    "Religiously_Unaffiliated"
]

majority_data3["majority_religion"] = majority_data3[religion_cols].idxmax(axis=1)

In [62]:
# year is 2020 for data

majority_filtered = majority_data3[[
    "Region",
    "country",
    "Diversity_rank",
    "RDI_score",
    "Diveristy_level",
    "Buddhists",
    "Christians",
    "Hindus",
    "Jews",
    "Muslims",
    "Other_Religions",
    "Religiously_Unaffiliated",
    "majority_religion"
]]

In [65]:
## justify these numbers using literature in paper

religion_traits = pd.DataFrame({
    "majority_religion": [
        "Christians",
        "Muslims",
        "Hindus",
        "Buddhists",
        "Jews",
        "Religiously_Unaffiliated",
        "Other_Religions"
    ],
    "hierarchy": [4, 4, 3, 2, 4, 1, 2],
    "exclusivity": [4, 4, 3, 2, 4, 1, 2],
    "collectivism": [3, 5, 4, 3, 4, 1, 3],
    "political_theology": [3, 5, 3, 2, 4, 1, 2],
    "pluralism_tolerance": [3, 2, 3, 4, 3, 5, 3]
})

In [66]:
final_data_trait = final_data2.merge(
    religion_traits,
    on="majority_religion",
    how="left"
)

In [67]:
religious_states = final_data2[
    final_data2["state-sponsored religion index"] == 2
].copy()

In [68]:
trait_data = religious_states.merge(
    religion_traits,
    on="majority_religion",
    how="left"
)

In [69]:
print(final_data_trait.head())

       country  year  intrastate_deaths  civilian_deaths  \
0  Afghanistan  1989               5174              303   
1  Afghanistan  1990               1478              101   
2  Afghanistan  1991               3302               38   
3  Afghanistan  1992               4287             1687   
4  Afghanistan  1993               4071              611   

   state-sponsored religion index        Region  Diversity_rank  RDI_score  \
0                             2.0  Asia-Pacific           200.0   0.032175   
1                             2.0  Asia-Pacific           200.0   0.032175   
2                             2.0  Asia-Pacific           200.0   0.032175   
3                             2.0  Asia-Pacific           200.0   0.032175   
4                             2.0  Asia-Pacific           200.0   0.032175   

  Diveristy_level  Buddhists  ...      Jews    Muslims  Other_Religions  \
0        Very low   0.020001  ...  0.000026  99.861969         0.090043   
1        Very low   

In [70]:
final_data_trait.to_csv("../data/final.csv", index=False)